In [ ]:
import requests
import os
import time
import json

In [ ]:
# ── Configuration — change these two lines to scrape any Chess.com user ──────
USERNAME   = "jf4bes"          # Chess.com username
OUTPUT_DIR = "games"           # folder to save into — use "games_sf" for sfink37
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://api.chess.com/pub/player"
HEADERS  = {"User-Agent": "chessanalysis/1.0 jeffreyforbes01@gmail.com"}

In [ ]:
# Fetch all monthly archive URLs
r = requests.get(f"{BASE_URL}/{USERNAME}/games/archives", headers=HEADERS)
r.raise_for_status()
archives = r.json()["archives"]
print(f"Found {len(archives)} months of games for {USERNAME}")
archives

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

total_games = 0

for archive_url in archives:
    parts = archive_url.rstrip("/").split("/")
    year, month = int(parts[-2]), int(parts[-1])

    print(f"Downloading {year}-{month:02d}...", end=" ", flush=True)
    try:
        r = requests.get(archive_url, headers=HEADERS)
        r.raise_for_status()
        games = r.json()["games"]

        # Save PGN
        pgn_path = os.path.join(OUTPUT_DIR, f"{year}_{month:02d}.pgn")
        with open(pgn_path, "w", encoding="utf-8") as f:
            for game in games:
                if game.get("pgn"):
                    f.write(game["pgn"] + "\n\n")

        # Save JSON
        json_path = os.path.join(OUTPUT_DIR, f"{year}_{month:02d}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(games, f, indent=2)

        print(f"{len(games)} games saved")
        total_games += len(games)
    except Exception as e:
        print(f"ERROR: {e}")

    time.sleep(0.5)

print(f"\nDone! {total_games} total games saved to ./{OUTPUT_DIR}/")